# 06 — Model Tuning And Improvement

**Pipeline role.** Sixth notebook. Applies cross-validated hyperparameter search, optional threshold tuning, and preprocessing/feature variants to the shortlisted models from notebook 05, under strict experiment-log discipline.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 3 — Model Building ("parameter tuning" effort explicitly mentioned in the rubric).
- Report Section 4 — Performance Evaluation: documents the selection criterion for the final model.

**TL;DR for teammates.** Every tuning run becomes one row in the experiment log. No fishing for the best number. A short, motivated search is worth more than a large aimless one. The test set stays untouched.


## Inputs, Outputs, Artifacts

**Inputs.**
- Shortlist and baseline results from notebook 05.
- `data/processed/X_train_feat_v1.*`, `y_train_v1.*` from notebook 04.
- [../reports/tables/experiment_log_template.csv](../reports/tables/experiment_log_template.csv) — extend this as the working experiment log.

**Outputs.**
- One experiment-log row per non-trivial run (search config, variant, threshold change, etc.).
- `reports/tables/tuning_results_v1.csv` — consolidated tuning comparison.
- `reports/figures/cv_metric_curves.png` (optional) — e.g., validation metric vs key hyperparameter.
- A single selected final model specification (model, hyperparameters, feature-set version, preprocessing version, decision threshold).

**Downstream consumers.**
- Notebook 07 re-trains the finalized model spec on full training data and evaluates on the held-out test set.
- Notebook 08 pulls the tuning comparison and selection rationale into the report.


## Methodological Influences

Hyperparameter tuning follows standard ISOM3360 cross-validation practice: motivated search spaces, leak-free `Pipeline` search, and one experiment-log row per run. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Key Questions To Answer Here

1. What is the CV protocol, and is it identical across tuning runs?
2. What is the search space per shortlisted model, and why that space?
3. Does class imbalance handling help CV metrics, and at what cost?
4. Does a non-default decision threshold improve F1 on validation folds?
5. Do alternative preprocessing/feature variants change the ranking of models?
6. What is the selection rule, and which configuration wins?


## Work Plan

### 6.1 CV Strategy (Fixed)
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` on the training set.
- Primary metric: ROC-AUC. Secondary: F1, precision, recall, accuracy.
- Use `scoring="roc_auc"` in `GridSearchCV` because ROC-AUC is threshold-invariant and therefore a fair comparator across candidates with different operating points. Threshold choice is a separate decision (Section 6.5) once the best configuration is selected / `RandomizedSearchCV` (scikit-learn).

### 6.2 Search Space Per Shortlisted Model
Keep spaces small and motivated. Typical starting points:

- **Logistic Regression.** `C ∈ {0.01, 0.1, 1, 10}`, `penalty ∈ {"l1","l2"}`, `class_weight ∈ {None, "balanced"}`.
- **Random Forest.** `n_estimators ∈ {200, 500}`, `max_depth ∈ {None, 10, 20}`, `min_samples_leaf ∈ {1, 5, 20}`, `class_weight ∈ {None, "balanced"}`.
- **Gradient Boosting / HistGradientBoosting.** `learning_rate ∈ {0.05, 0.1}`, `max_depth ∈ {3, 6}`, `max_iter` or `n_estimators` ∈ {200, 500}.

For each search, use `Pipeline` so preprocessing fits within each CV fold.

### 6.3 Optional: Class Imbalance Handling
- Try `class_weight="balanced"` where supported.
- Try resampling (e.g., `RandomUnderSampler`) **inside CV folds only** to avoid leakage.
- Log resampling variants as separate experiment rows.

### 6.4 Optional: Threshold Tuning
- Sweep decision thresholds on validation folds to optimize F1 (or a business-oriented cost function).
- Record the chosen threshold in the experiment log row.

### 6.5 Optional: Preprocessing / Feature Variants
- Run one or two variants (e.g., `feature_set_v1` vs a scaled-only variant, or `prep_v1` vs `prep_v2` with different imputation).
- Tag each run with `feature_set_version` and `preprocessing_version` to keep comparisons traceable.

### 6.6 Selection Rule
- State the selection rule up front, e.g.: "pick the highest mean CV ROC-AUC; break ties with F1; penalize >2σ training/validation gaps as likely overfitting".
- Apply the rule. Document the chosen model, hyperparameters, threshold, feature version, preprocessing version.

### 6.7 Persist Artifacts
- Save consolidated tuning table to `reports/tables/tuning_results_v1.csv`.
- Append tuning rows to the experiment log.
- Record the final model spec in a clearly labelled cell.


## Implementation Block 6.1 — Setup

This block implements the setup boundary for Notebook 06: load the frozen `feature_set_v1` training artifacts, load the saved baseline outputs from Notebook 05, and confirm the shortlist that will be tuned under one shared validation protocol.

No hyperparameter search is run in this step. The purpose is to establish a stable, versioned tuning context before the CV strategy and search spaces are defined.

The setup follows:
- [../Group Project Guideline.md](../Group%20Project%20Guideline.md) — tuning comparisons should be reproducible and tied to saved artifacts.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

FEATURE_SET_VERSION = "feature_set_v1"
ARTIFACT_VERSION = "v1"
RANDOM_STATE = 42
TARGET_COLUMN = "is_canceled"
NOTEBOOK_STAGE = "06_model_tuning_and_improvement"

PROCESSED_DATA_CANDIDATES = [
    Path("../data/processed"),
    Path("data/processed"),
]
REPORTS_TABLES_CANDIDATES = [
    Path("../reports/tables"),
    Path("reports/tables"),
]

PROCESSED_DATA_DIR = next(
    (path for path in PROCESSED_DATA_CANDIDATES if path.exists()),
    PROCESSED_DATA_CANDIDATES[0],
)
REPORTS_TABLES_DIR = next(
    (path for path in REPORTS_TABLES_CANDIDATES if path.exists()),
    REPORTS_TABLES_CANDIDATES[0],
)

if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError("The processed data directory containing feature_set_v1 was not found.")
if not REPORTS_TABLES_DIR.exists():
    raise FileNotFoundError("The reports/tables directory containing Notebook 05 outputs was not found.")

X_train_feat_path = PROCESSED_DATA_DIR / f"X_train_feat_{ARTIFACT_VERSION}.csv"
y_train_path = PROCESSED_DATA_DIR / f"y_train_{ARTIFACT_VERSION}.csv"
baseline_results_path = REPORTS_TABLES_DIR / f"baseline_results_{ARTIFACT_VERSION}.csv"
experiment_log_working_path = REPORTS_TABLES_DIR / "experiment_log_working.csv"

required_paths = [
    X_train_feat_path,
    y_train_path,
    baseline_results_path,
    experiment_log_working_path,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing expected Notebook 05 or processed-data artifacts: {missing_paths}")

X_train_feat = pd.read_csv(X_train_feat_path)
y_train = pd.read_csv(y_train_path)[TARGET_COLUMN]
baseline_results = pd.read_csv(baseline_results_path)
experiment_log_working = pd.read_csv(experiment_log_working_path)

if len(X_train_feat) != len(y_train):
    raise ValueError("X_train_feat and y_train row counts do not match.")

shortlist_for_tuning = (
    baseline_results.loc[baseline_results["model_name"] != "Dummy Most Frequent", ["model_name", "roc_auc", "family", "notes"]]
    .sort_values("roc_auc", ascending=False)
    .head(3)
    .reset_index(drop=True)
    .copy()
)

tuning_setup_summary = pd.DataFrame(
    {
        "check": [
            "processed_data_dir",
            "reports_tables_dir",
            "feature_set_version",
            "X_train_feat_shape",
            "y_train_rows",
            "train_cancellation_rate_pct",
            "baseline_rows_loaded",
            "experiment_log_rows_loaded",
            "shortlist_size",
            "random_state",
        ],
        "value": [
            str(PROCESSED_DATA_DIR.resolve()),
            str(REPORTS_TABLES_DIR.resolve()),
            FEATURE_SET_VERSION,
            str(X_train_feat.shape),
            len(y_train),
            round(y_train.mean() * 100, 2),
            len(baseline_results),
            len(experiment_log_working),
            len(shortlist_for_tuning),
            RANDOM_STATE,
        ],
    }
)

display(tuning_setup_summary)
display(shortlist_for_tuning)

,check,value
0,processed_data_dir,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,reports_tables_dir,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
2,feature_set_version,feature_set_v1
3,X_train_feat_shape,"(69577, 44)"
4,y_train_rows,69577
5,train_cancellation_rate_pct,27.31
6,baseline_rows_loaded,5
7,experiment_log_rows_loaded,5
8,shortlist_size,3
9,random_state,42


,model_name,roc_auc,family,notes
0,Random Forest,0.901858,tree_ensemble,Modest ensemble baseline with default depth be...
1,Logistic Regression,0.864123,linear,Scale-sensitive linear baseline with default d...
2,K-Nearest Neighbors,0.825733,distance,Scale-sensitive neighborhood baseline for cont...


## Implementation Block 6.2 — Shared CV Strategy

This block fixes the common validation protocol for all tuning work in Notebook 06: identify numeric and categorical feature groups, define one shared 5-fold stratified CV object, and create reusable preprocessing builders that can be attached to each model-specific search pipeline.

No hyperparameter search is run in this step. The purpose is to lock the comparison rules before any search space is introduced, so tuned runs remain directly comparable to one another.

The protocol keeps ROC-AUC as the primary selection metric, retains F1, precision, recall, and accuracy as secondary diagnostics, and preserves `random_state=42` throughout.


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_feature_columns = X_train_feat.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_feature_columns = [
    column for column in X_train_feat.columns if column not in categorical_feature_columns
]

cv_protocol = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
primary_metric = "roc_auc"
secondary_metrics = ["f1", "precision", "recall", "accuracy"]
scoring = {metric: metric for metric in [primary_metric, *secondary_metrics]}
selection_rule = "Highest mean CV ROC-AUC; break ties with F1; keep threshold tuning separate from model search; never use the held-out test set here."

def build_tuning_preprocessor(scale_numeric):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler() if scale_numeric else "passthrough"),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_feature_columns),
            ("categorical", categorical_transformer, categorical_feature_columns),
        ],
        remainder="drop",
    )

cv_protocol_summary = pd.DataFrame(
    {
        "setting": [
            "cv_strategy",
            "cv_folds",
            "primary_metric",
            "secondary_metrics",
            "numeric_feature_count",
            "categorical_feature_count",
            "selection_rule",
        ],
        "value": [
            "StratifiedKFold",
            cv_protocol.get_n_splits(),
            primary_metric,
            ";".join(secondary_metrics),
            len(numeric_feature_columns),
            len(categorical_feature_columns),
            selection_rule,
        ],
    }
)

display(cv_protocol_summary)

,setting,value
0,cv_strategy,StratifiedKFold
1,cv_folds,5
2,primary_metric,roc_auc
3,secondary_metrics,f1;precision;recall;accuracy
4,numeric_feature_count,32
5,categorical_feature_count,12
6,selection_rule,Highest mean CV ROC-AUC; break ties with F1; k...


## Implementation Block 6.3 — Search Space Per Shortlisted Model

This block defines the initial search spaces for the three shortlisted baseline models from Notebook 05: Random Forest, Logistic Regression, and K-Nearest Neighbors.

No search is executed here. The purpose is to make the tuning scope explicit, small, and defensible before any cross-validated model fitting begins.

Each space is intentionally modest: it expands around the untuned baseline, covers a small number of high-impact hyperparameters, and keeps preprocessing choices visible through the shared pipeline structure defined in the previous block.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.neighbors import KNeighborsClassifier

shortlisted_model_names = shortlist_for_tuning["model_name"].tolist()

tuning_model_specs = [
    {
        "model_name": "Logistic Regression",
        "family": "linear",
        "scale_numeric": True,
        "estimator": LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
        "param_grid": {
            "model__C": [0.01, 0.1, 1.0, 10.0],
            "model__penalty": ["l1", "l2"],
            "model__class_weight": [None, "balanced"],
        },
        "search_rationale": "Tune regularization strength, penalty type, and imbalance handling around the linear baseline.",
    },
    {
        "model_name": "Random Forest",
        "family": "tree_ensemble",
        "scale_numeric": False,
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "param_grid": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 5, 20],
            "model__class_weight": [None, "balanced"],
        },
        "search_rationale": "Tune ensemble size, tree depth, leaf size, and class weighting around the strongest baseline.",
    },
    {
        "model_name": "K-Nearest Neighbors",
        "family": "distance",
        "scale_numeric": True,
        "estimator": KNeighborsClassifier(),
        "param_grid": {
            "model__n_neighbors": [5, 11, 21],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        "search_rationale": "Tune neighborhood size, vote weighting, and distance metric for the scale-sensitive distance baseline.",
    },
]

tuning_model_specs = [
    spec for spec in tuning_model_specs if spec["model_name"] in shortlisted_model_names
]

if len(tuning_model_specs) != len(shortlisted_model_names):
    raise ValueError("The tuning spec registry does not fully match the shortlisted models.")

tuning_search_space_summary = pd.DataFrame(
    [
        {
            "model_name": spec["model_name"],
            "family": spec["family"],
            "scale_numeric": spec["scale_numeric"],
            "grid_configurations": len(list(ParameterGrid(spec["param_grid"]))),
            "hyperparameters": ", ".join(
                parameter_name.replace("model__", "") for parameter_name in spec["param_grid"].keys()
            ),
            "search_rationale": spec["search_rationale"],
        }
        for spec in tuning_model_specs
    ]
).sort_values("model_name").reset_index(drop=True)

display(tuning_search_space_summary)

,model_name,family,scale_numeric,grid_configurations,hyperparameters,search_rationale
0,K-Nearest Neighbors,distance,True,12,"n_neighbors, weights, p","Tune neighborhood size, vote weighting, and di..."
1,Logistic Regression,linear,True,16,"C, penalty, class_weight","Tune regularization strength, penalty type, an..."
2,Random Forest,tree_ensemble,False,36,"n_estimators, max_depth, min_samples_leaf, cla...","Tune ensemble size, tree depth, leaf size, and..."


## Implementation Block 6.4 — Tune Logistic Regression

This block runs the first full cross-validated tuning search in Notebook 06, limited to Logistic Regression. The shared preprocessing builder, fixed CV protocol, and explicit search grid from the earlier blocks are used without modification.

Only one shortlisted model is tuned here. The purpose is to validate the search workflow end-to-end on the simplest model family before repeating the same pattern for the other shortlisted candidates.

The held-out test set remains untouched. All metrics reported below are cross-validated training-split metrics from `GridSearchCV`.


In [5]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

logistic_tuning_spec = next(
    spec for spec in tuning_model_specs if spec["model_name"] == "Logistic Regression"
 )

logistic_tuning_pipeline = Pipeline(
    steps=[
        ("preprocess", build_tuning_preprocessor(scale_numeric=logistic_tuning_spec["scale_numeric"])),
        ("model", clone(logistic_tuning_spec["estimator"])),
    ]
)

logistic_tuning_search = GridSearchCV(
    estimator=logistic_tuning_pipeline,
    param_grid=logistic_tuning_spec["param_grid"],
    scoring=scoring,
    refit=primary_metric,
    cv=cv_protocol,
    n_jobs=1,
    return_train_score=True,
)
logistic_tuning_search.fit(X_train_feat, y_train)

logistic_cv_results = (
    pd.DataFrame(logistic_tuning_search.cv_results_)
    .sort_values([f"rank_test_{primary_metric}", f"mean_test_{primary_metric}"], ascending=[True, False])
    .reset_index(drop=True)
)

best_logistic_result = logistic_cv_results.iloc[0]
best_logistic_params = logistic_tuning_search.best_params_

logistic_tuning_summary = pd.DataFrame(
    {
        "field": [
            "model_name",
            "best_rank",
            "best_roc_auc",
            "best_f1",
            "best_precision",
            "best_recall",
            "best_accuracy",
            "grid_configurations",
            "best_params",
        ],
        "value": [
            logistic_tuning_spec["model_name"],
            int(best_logistic_result[f"rank_test_{primary_metric}"]),
            round(best_logistic_result[f"mean_test_{primary_metric}"], 6),
            round(best_logistic_result["mean_test_f1"], 6),
            round(best_logistic_result["mean_test_precision"], 6),
            round(best_logistic_result["mean_test_recall"], 6),
            round(best_logistic_result["mean_test_accuracy"], 6),
            len(logistic_cv_results),
            "; ".join(f"{key}={value}" for key, value in best_logistic_params.items()),
        ],
    }
)

logistic_tuning_top_configs = logistic_cv_results.loc[
    :,
    [
        "rank_test_roc_auc",
        "mean_test_roc_auc",
        "mean_test_f1",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_accuracy",
        "param_model__C",
        "param_model__penalty",
        "param_model__class_weight",
    ],
] .head(5)

if "tuning_best_results" not in globals():
    tuning_best_results = {}
tuning_best_results[logistic_tuning_spec["model_name"]] = {
    "model_name": logistic_tuning_spec["model_name"],
    "family": logistic_tuning_spec["family"],
    "scale_numeric": logistic_tuning_spec["scale_numeric"],
    "best_params": best_logistic_params,
    "roc_auc": float(best_logistic_result[f"mean_test_{primary_metric}"]),
    "f1": float(best_logistic_result["mean_test_f1"]),
    "precision": float(best_logistic_result["mean_test_precision"]),
    "recall": float(best_logistic_result["mean_test_recall"]),
    "accuracy": float(best_logistic_result["mean_test_accuracy"]),
}

display(logistic_tuning_summary)
display(logistic_tuning_top_configs)

,field,value
0,model_name,Logistic Regression
1,best_rank,1
2,best_roc_auc,0.864169
3,best_f1,0.610522
4,best_precision,0.700967
5,best_recall,0.540866
6,best_accuracy,0.811547
7,grid_configurations,16
8,best_params,model__C=1.0; model__class_weight=None; model_...


,rank_test_roc_auc,mean_test_roc_auc,mean_test_f1,mean_test_precision,mean_test_recall,mean_test_accuracy,param_model__C,param_model__penalty,param_model__class_weight
0,1,0.864169,0.610522,0.700967,0.540866,0.811547,1.0,l2,None
1,2,0.864091,0.610252,0.701566,0.540077,0.811604,1.0,l1,None
2,3,0.864086,0.610975,0.700305,0.541971,0.811518,10.0,l2,None
3,4,0.864023,0.610916,0.700241,0.541919,0.811489,10.0,l1,None
4,5,0.863937,0.660862,0.559282,0.807589,0.773618,1.0,l2,balanced


## Implementation Block 6.5 — Tune Random Forest

This block runs the second full cross-validated tuning search in Notebook 06, limited to Random Forest. It reuses the same shared preprocessing builder, fixed CV protocol, and explicit search grid defined earlier so the result remains directly comparable with the Logistic Regression tuning run.

Only one shortlisted model is tuned here. The purpose is to test whether the strongest untuned baseline can be improved through a small, disciplined tree-ensemble search without touching the held-out test set.

All reported metrics remain cross-validated training-split metrics from `GridSearchCV`.


In [6]:
random_forest_tuning_spec = next(
    spec for spec in tuning_model_specs if spec["model_name"] == "Random Forest"
 )

random_forest_tuning_pipeline = Pipeline(
    steps=[
        ("preprocess", build_tuning_preprocessor(scale_numeric=random_forest_tuning_spec["scale_numeric"])),
        ("model", clone(random_forest_tuning_spec["estimator"])),
    ]
)

random_forest_tuning_search = GridSearchCV(
    estimator=random_forest_tuning_pipeline,
    param_grid=random_forest_tuning_spec["param_grid"],
    scoring=scoring,
    refit=primary_metric,
    cv=cv_protocol,
    n_jobs=1,
    return_train_score=True,
)
random_forest_tuning_search.fit(X_train_feat, y_train)

random_forest_cv_results = (
    pd.DataFrame(random_forest_tuning_search.cv_results_)
    .sort_values([f"rank_test_{primary_metric}", f"mean_test_{primary_metric}"], ascending=[True, False])
    .reset_index(drop=True)
)

best_random_forest_result = random_forest_cv_results.iloc[0]
best_random_forest_params = random_forest_tuning_search.best_params_

random_forest_tuning_summary = pd.DataFrame(
    {
        "field": [
            "model_name",
            "best_rank",
            "best_roc_auc",
            "best_f1",
            "best_precision",
            "best_recall",
            "best_accuracy",
            "grid_configurations",
            "best_params",
        ],
        "value": [
            random_forest_tuning_spec["model_name"],
            int(best_random_forest_result[f"rank_test_{primary_metric}"]),
            round(best_random_forest_result[f"mean_test_{primary_metric}"], 6),
            round(best_random_forest_result["mean_test_f1"], 6),
            round(best_random_forest_result["mean_test_precision"], 6),
            round(best_random_forest_result["mean_test_recall"], 6),
            round(best_random_forest_result["mean_test_accuracy"], 6),
            len(random_forest_cv_results),
            "; ".join(f"{key}={value}" for key, value in best_random_forest_params.items()),
        ],
    }
)

random_forest_tuning_top_configs = random_forest_cv_results.loc[
    :,
    [
        "rank_test_roc_auc",
        "mean_test_roc_auc",
        "mean_test_f1",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_accuracy",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__class_weight",
    ],
] .head(5)

tuning_best_results[random_forest_tuning_spec["model_name"]] = {
    "model_name": random_forest_tuning_spec["model_name"],
    "family": random_forest_tuning_spec["family"],
    "scale_numeric": random_forest_tuning_spec["scale_numeric"],
    "best_params": best_random_forest_params,
    "roc_auc": float(best_random_forest_result[f"mean_test_{primary_metric}"]),
    "f1": float(best_random_forest_result["mean_test_f1"]),
    "precision": float(best_random_forest_result["mean_test_precision"]),
    "recall": float(best_random_forest_result["mean_test_recall"]),
    "accuracy": float(best_random_forest_result["mean_test_accuracy"]),
}

display(random_forest_tuning_summary)
display(random_forest_tuning_top_configs)

,field,value
0,model_name,Random Forest
1,best_rank,1
2,best_roc_auc,0.9061
3,best_f1,0.691639
4,best_precision,0.77794
5,best_recall,0.622599
6,best_accuracy,0.848398
7,grid_configurations,36
8,best_params,model__class_weight=balanced; model__max_depth...


,rank_test_roc_auc,mean_test_roc_auc,mean_test_f1,mean_test_precision,mean_test_recall,mean_test_accuracy,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__class_weight
0,1,0.906100,0.691639,0.777940,0.622599,0.848398,500,None,1,balanced
1,2,0.904827,0.689819,0.779882,0.618441,0.848125,200,None,1,balanced
2,3,0.904811,0.693833,0.774986,0.628072,0.848628,500,None,1,None
3,4,0.904745,0.719836,0.664903,0.784696,0.833192,500,None,5,balanced
4,5,0.904296,0.718436,0.662559,0.784643,0.832042,200,None,5,balanced


## Implementation Block 6.6 — Tune K-Nearest Neighbors

This block runs the third shortlisted-model tuning search in Notebook 06, limited to K-Nearest Neighbors. It reuses the same shared preprocessing builder, fixed CV protocol, and explicit search grid so the result stays directly comparable with the Logistic Regression and Random Forest tuning runs.

Only one shortlisted model is tuned here. The purpose is to complete the like-for-like comparison across all shortlisted candidates before any final model selection or artifact persistence happens.

All reported metrics remain cross-validated training-split metrics from `GridSearchCV`, and the held-out test set remains untouched.


In [7]:
knn_tuning_spec = next(
    spec for spec in tuning_model_specs if spec["model_name"] == "K-Nearest Neighbors"
 )

knn_tuning_pipeline = Pipeline(
    steps=[
        ("preprocess", build_tuning_preprocessor(scale_numeric=knn_tuning_spec["scale_numeric"])),
        ("model", clone(knn_tuning_spec["estimator"])),
    ]
)

knn_tuning_search = GridSearchCV(
    estimator=knn_tuning_pipeline,
    param_grid=knn_tuning_spec["param_grid"],
    scoring=scoring,
    refit=primary_metric,
    cv=cv_protocol,
    n_jobs=1,
    return_train_score=True,
)
knn_tuning_search.fit(X_train_feat, y_train)

knn_cv_results = (
    pd.DataFrame(knn_tuning_search.cv_results_)
    .sort_values([f"rank_test_{primary_metric}", f"mean_test_{primary_metric}"], ascending=[True, False])
    .reset_index(drop=True)
)

best_knn_result = knn_cv_results.iloc[0]
best_knn_params = knn_tuning_search.best_params_

knn_tuning_summary = pd.DataFrame(
    {
        "field": [
            "model_name",
            "best_rank",
            "best_roc_auc",
            "best_f1",
            "best_precision",
            "best_recall",
            "best_accuracy",
            "grid_configurations",
            "best_params",
        ],
        "value": [
            knn_tuning_spec["model_name"],
            int(best_knn_result[f"rank_test_{primary_metric}"]),
            round(best_knn_result[f"mean_test_{primary_metric}"], 6),
            round(best_knn_result["mean_test_f1"], 6),
            round(best_knn_result["mean_test_precision"], 6),
            round(best_knn_result["mean_test_recall"], 6),
            round(best_knn_result["mean_test_accuracy"], 6),
            len(knn_cv_results),
            "; ".join(f"{key}={value}" for key, value in best_knn_params.items()),
        ],
    }
)

knn_tuning_top_configs = knn_cv_results.loc[
    :,
    [
        "rank_test_roc_auc",
        "mean_test_roc_auc",
        "mean_test_f1",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_accuracy",
        "param_model__n_neighbors",
        "param_model__weights",
        "param_model__p",
    ],
] .head(5)

tuning_best_results[knn_tuning_spec["model_name"]] = {
    "model_name": knn_tuning_spec["model_name"],
    "family": knn_tuning_spec["family"],
    "scale_numeric": knn_tuning_spec["scale_numeric"],
    "best_params": best_knn_params,
    "roc_auc": float(best_knn_result[f"mean_test_{primary_metric}"]),
    "f1": float(best_knn_result["mean_test_f1"]),
    "precision": float(best_knn_result["mean_test_precision"]),
    "recall": float(best_knn_result["mean_test_recall"]),
    "accuracy": float(best_knn_result["mean_test_accuracy"]),
}

display(knn_tuning_summary)
display(knn_tuning_top_configs)

,field,value
0,model_name,K-Nearest Neighbors
1,best_rank,1
2,best_roc_auc,0.853526
3,best_f1,0.642833
4,best_precision,0.692686
5,best_recall,0.599706
6,best_accuracy,0.818029
7,grid_configurations,12
8,best_params,model__n_neighbors=21; model__p=1; model__weig...


,rank_test_roc_auc,mean_test_roc_auc,mean_test_f1,mean_test_precision,mean_test_recall,mean_test_accuracy,param_model__n_neighbors,param_model__weights,param_model__p
0,1,0.853526,0.642833,0.692686,0.599706,0.818029,21,distance,1
1,2,0.851619,0.630726,0.687707,0.582496,0.813746,21,uniform,1
2,3,0.850827,0.632603,0.688569,0.585075,0.814421,21,distance,2
3,4,0.850710,0.624895,0.686806,0.573233,0.812064,21,uniform,2
4,5,0.847795,0.642849,0.671694,0.616442,0.812955,11,distance,1


## Implementation Block 6.7 — Selection Rule And Provisional Winner

This block consolidates the best tuned result from each shortlisted model family, compares each tuned model against its Notebook 05 baseline, and applies the previously stated selection rule to identify a single provisional winner for Notebook 07.

No files are written in this step. The purpose is to make the model-selection decision explicit before artifact persistence and handoff updates are completed.

The rule used here is the same one fixed earlier in this notebook: choose the highest mean CV ROC-AUC, break ties with F1, and keep the held-out test set untouched until Notebook 07.


In [8]:
tuning_results_summary = (
    pd.DataFrame(tuning_best_results.values())
    .sort_values(["roc_auc", "f1"], ascending=[False, False])
    .reset_index(drop=True)
)

baseline_lookup = baseline_results.set_index("model_name")
tuning_results_summary["baseline_roc_auc"] = tuning_results_summary["model_name"].map(baseline_lookup["roc_auc"])
tuning_results_summary["roc_auc_gain_vs_baseline"] = (
    tuning_results_summary["roc_auc"] - tuning_results_summary["baseline_roc_auc"]
)

tuning_results_summary["baseline_f1"] = tuning_results_summary["model_name"].map(baseline_lookup["f1"])
tuning_results_summary["f1_gain_vs_baseline"] = (
    tuning_results_summary["f1"] - tuning_results_summary["baseline_f1"]
)

selection_columns = [
    "model_name",
    "family",
    "roc_auc",
    "baseline_roc_auc",
    "roc_auc_gain_vs_baseline",
    "f1",
    "baseline_f1",
    "f1_gain_vs_baseline",
    "precision",
    "recall",
    "accuracy",
    "scale_numeric",
    "best_params",
]

tuning_selection_table = tuning_results_summary.loc[:, selection_columns].copy()
tuning_selection_table[[
    "roc_auc",
    "baseline_roc_auc",
    "roc_auc_gain_vs_baseline",
    "f1",
    "baseline_f1",
    "f1_gain_vs_baseline",
    "precision",
    "recall",
    "accuracy",
]] = tuning_selection_table[[
    "roc_auc",
    "baseline_roc_auc",
    "roc_auc_gain_vs_baseline",
    "f1",
    "baseline_f1",
    "f1_gain_vs_baseline",
    "precision",
    "recall",
    "accuracy",
]].round(6)

selected_model_row = tuning_results_summary.iloc[0]
provisional_final_model_spec = {
    "model_name": selected_model_row["model_name"],
    "family": selected_model_row["family"],
    "feature_set_version": FEATURE_SET_VERSION,
    "preprocessing_version": ARTIFACT_VERSION,
    "decision_threshold": 0.5,
    "primary_metric": primary_metric,
    "selection_rule": selection_rule,
    "scale_numeric": bool(selected_model_row["scale_numeric"]),
    "best_params": selected_model_row["best_params"],
    "roc_auc": float(selected_model_row["roc_auc"]),
    "f1": float(selected_model_row["f1"]),
    "precision": float(selected_model_row["precision"]),
    "recall": float(selected_model_row["recall"]),
    "accuracy": float(selected_model_row["accuracy"]),
}

provisional_final_model_summary = pd.DataFrame(
    {
        "field": [
            "selected_model",
            "family",
            "feature_set_version",
            "preprocessing_version",
            "decision_threshold",
            "primary_metric",
            "selection_rule",
            "selected_roc_auc",
            "selected_f1",
            "selected_precision",
            "selected_recall",
            "selected_accuracy",
            "best_params",
        ],
        "value": [
            provisional_final_model_spec["model_name"],
            provisional_final_model_spec["family"],
            provisional_final_model_spec["feature_set_version"],
            provisional_final_model_spec["preprocessing_version"],
            provisional_final_model_spec["decision_threshold"],
            provisional_final_model_spec["primary_metric"],
            provisional_final_model_spec["selection_rule"],
            round(provisional_final_model_spec["roc_auc"], 6),
            round(provisional_final_model_spec["f1"], 6),
            round(provisional_final_model_spec["precision"], 6),
            round(provisional_final_model_spec["recall"], 6),
            round(provisional_final_model_spec["accuracy"], 6),
            "; ".join(
                f"{key}={value}" for key, value in provisional_final_model_spec["best_params"].items()
            ),
        ],
    }
)

display(tuning_selection_table)
display(provisional_final_model_summary)

,model_name,family,roc_auc,baseline_roc_auc,roc_auc_gain_vs_baseline,f1,baseline_f1,f1_gain_vs_baseline,precision,recall,accuracy,scale_numeric,best_params
0,Random Forest,tree_ensemble,0.906100,0.901858,0.004242,0.691639,0.688566,0.003074,0.777940,0.622599,0.848398,False,"{'model__class_weight': 'balanced', 'model__ma..."
1,Logistic Regression,linear,0.864169,0.864123,0.000046,0.610522,0.610623,-0.000101,0.700967,0.540866,0.811547,True,"{'model__C': 1.0, 'model__class_weight': None,..."
2,K-Nearest Neighbors,distance,0.853526,0.825733,0.027793,0.642833,0.619576,0.023257,0.692686,0.599706,0.818029,True,"{'model__n_neighbors': 21, 'model__p': 1, 'mod..."


,field,value
0,selected_model,Random Forest
1,family,tree_ensemble
2,feature_set_version,feature_set_v1
3,preprocessing_version,v1
4,decision_threshold,0.5
5,primary_metric,roc_auc
6,selection_rule,Highest mean CV ROC-AUC; break ties with F1; k...
7,selected_roc_auc,0.9061
8,selected_f1,0.691639
9,selected_precision,0.77794


## Implementation Block 6.8 — Persist Tuning Artifacts

This block writes the key Notebook 06 outputs to disk: the consolidated tuning comparison table, a machine-readable final model specification table, and an updated working experiment log containing the tuning-stage rows.

No new model fitting is performed here. The purpose is to freeze the tuned-model comparison and handoff artifacts before the checklist and notebook-tail summaries are updated.

The persistence step is idempotent: if this block is rerun, earlier Notebook 06 tuning rows are replaced rather than duplicated in the working experiment log.


In [9]:
from datetime import date

tuning_results_path = REPORTS_TABLES_DIR / f"tuning_results_{ARTIFACT_VERSION}.csv"
final_model_spec_path = REPORTS_TABLES_DIR / f"final_model_spec_{ARTIFACT_VERSION}.csv"
updated_experiment_log_path = REPORTS_TABLES_DIR / "experiment_log_working.csv"

tuning_results_to_save = tuning_selection_table.copy()
tuning_results_to_save["best_params"] = tuning_results_to_save["best_params"].apply(
    lambda params: "; ".join(f"{key}={value}" for key, value in params.items())
)
tuning_results_to_save.to_csv(tuning_results_path, index=False)

final_model_spec_table = pd.DataFrame(
    [
        {
            "model_name": provisional_final_model_spec["model_name"],
            "family": provisional_final_model_spec["family"],
            "feature_set_version": provisional_final_model_spec["feature_set_version"],
            "preprocessing_version": provisional_final_model_spec["preprocessing_version"],
            "decision_threshold": provisional_final_model_spec["decision_threshold"],
            "primary_metric": provisional_final_model_spec["primary_metric"],
            "selection_rule": provisional_final_model_spec["selection_rule"],
            "scale_numeric": provisional_final_model_spec["scale_numeric"],
            "roc_auc": round(provisional_final_model_spec["roc_auc"], 6),
            "f1": round(provisional_final_model_spec["f1"], 6),
            "precision": round(provisional_final_model_spec["precision"], 6),
            "recall": round(provisional_final_model_spec["recall"], 6),
            "accuracy": round(provisional_final_model_spec["accuracy"], 6),
            "best_params": "; ".join(
                f"{key}={value}" for key, value in provisional_final_model_spec["best_params"].items()
            ),
        }
    ]
)
final_model_spec_table.to_csv(final_model_spec_path, index=False)

run_date = date.today().isoformat()
tuning_log_rows = []
for run_number, row in tuning_results_summary.reset_index(drop=True).iterrows():
    tuning_log_rows.append(
        {
            "run_id": f"tuning_v1_{run_number + 1:02d}",
            "run_date": run_date,
            "notebook_stage": NOTEBOOK_STAGE,
            "model_name": row["model_name"],
            "feature_set_version": FEATURE_SET_VERSION,
            "preprocessing_version": ARTIFACT_VERSION,
            "validation_strategy": "StratifiedKFold",
            "cv_folds": cv_protocol.get_n_splits(),
            "decision_threshold": 0.5,
            "random_state": RANDOM_STATE,
            "roc_auc": round(row["roc_auc"], 6),
            "f1": round(row["f1"], 6),
            "precision": round(row["precision"], 6),
            "recall": round(row["recall"], 6),
            "accuracy": round(row["accuracy"], 6),
            "primary_metric": primary_metric,
            "secondary_metrics": ";".join(secondary_metrics),
            "notes": (
                f"Best tuned configuration. Baseline ROC-AUC delta={row['roc_auc_gain_vs_baseline']:.6f}. "
                + "; ".join(f"{key}={value}" for key, value in row["best_params"].items())
            ),
        }
    )

tuning_log_rows.append(
    {
        "run_id": "tuning_v1_04",
        "run_date": run_date,
        "notebook_stage": NOTEBOOK_STAGE,
        "model_name": "Selected Final Spec",
        "feature_set_version": FEATURE_SET_VERSION,
        "preprocessing_version": ARTIFACT_VERSION,
        "validation_strategy": "StratifiedKFold",
        "cv_folds": cv_protocol.get_n_splits(),
        "decision_threshold": provisional_final_model_spec["decision_threshold"],
        "random_state": RANDOM_STATE,
        "roc_auc": round(provisional_final_model_spec["roc_auc"], 6),
        "f1": round(provisional_final_model_spec["f1"], 6),
        "precision": round(provisional_final_model_spec["precision"], 6),
        "recall": round(provisional_final_model_spec["recall"], 6),
        "accuracy": round(provisional_final_model_spec["accuracy"], 6),
        "primary_metric": primary_metric,
        "secondary_metrics": ";".join(secondary_metrics),
        "notes": (
            "Provisional final winner selected in Notebook 06. "
            + "; ".join(
                f"{key}={value}" for key, value in provisional_final_model_spec["best_params"].items()
            )
        ),
    }
)

experiment_log_columns = experiment_log_working.columns.tolist()
experiment_log_base = experiment_log_working.loc[
    ~experiment_log_working["run_id"].str.startswith("tuning_v1_", na=False)
].copy()
tuning_log_df = pd.DataFrame(tuning_log_rows)[experiment_log_columns]
experiment_log_working = pd.concat([experiment_log_base, tuning_log_df], ignore_index=True)
experiment_log_working.to_csv(updated_experiment_log_path, index=False)

tuning_artifact_summary = pd.DataFrame(
    {
        "artifact": [
            "tuning_results_v1",
            "final_model_spec_v1",
            "experiment_log_working",
        ],
        "path": [
            str(tuning_results_path.resolve()),
            str(final_model_spec_path.resolve()),
            str(updated_experiment_log_path.resolve()),
        ],
    }
)

display(tuning_artifact_summary)

,artifact,path
0,tuning_results_v1,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,final_model_spec_v1,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
2,experiment_log_working,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...


## Review Checklist

- [x] CV strategy is identical across tuning runs so comparisons are fair.
- [x] Search space for each shortlisted model is explicit and motivated.
- [x] Every meaningful run is logged in the working experiment log with Notebook 06 tuning rows replacing prior duplicates if rerun.
- [x] Threshold tuning was not performed, so no validation-threshold leakage was introduced.
- [x] Selection criterion is stated before picking the final model.
- [x] Final model spec is recorded in one place in the notebook and exported to `reports/tables/final_model_spec_v1.csv`.
- [x] The held-out test set has not been touched.


## Handoff To Notebook 07

- Frozen artifacts are ready: `tuning_results_v1.csv`, `final_model_spec_v1.csv`, and the updated `experiment_log_working.csv`.
- The provisional winning configuration is a Random Forest on `feature_set_v1` with preprocessing version `v1`, decision threshold `0.5`, and tuned parameters `class_weight=balanced`, `max_depth=None`, `min_samples_leaf=1`, `n_estimators=500`.
- Notebook 07 should retrain exactly this single specification on the full training set and evaluate it on the held-out test set exactly once.
- If Notebook 07 shows a material regression or calibration issue, return to Notebook 06 and revise the training-side selection logic rather than changing the test protocol.
